# T3-05 · Chọn silver pool và chuẩn bị bundles
Chạy trên Colab CPU. Notebook chọn ID, đọc MS1 và MS2 trực tiếp từ MS2_RESULTS.zip (không giải nén), rồi lưu bundle. Chưa gán nhãn. Nếu nguồn chỉ ở Được chia sẻ với tôi, thêm shortcut vào MyDrive trước.


In [ ]:
# @title 1. Kết nối Drive và lấy code từ nhánh T3-05
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys
from pathlib import Path
REPO = Path('/content/arxiv-data-science')
BRANCH = 'feature/m3-tv3-auto-labeling'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/LeoTkiet/arxiv-data-science.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[matching,reporting]'], check=True)
print('Code:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# @title 2. Chọn 2.000 bài và danh sách dự phòng
from arxiv_pipeline.matching.silver_pool import select_pool
POPULATION = Path('/content/drive/MyDrive/T3-05/population_manifest.json') # @param {type:'string'}
OUTPUT = Path('/content/drive/MyDrive/T3-05/output') # @param {type:'string'}
TARGET_COUNT = 2000 # @param {type:'integer'}
SEED = 42 # @param {type:'integer'}
selection = select_pool(Path(POPULATION), REPO/'configs/ms1_group6_ranges.json', Path(OUTPUT)/'selection.json', seed=SEED, target_count=TARGET_COUNT)
print('Đã chọn:', sum(map(len, selection['selected'].values())), '| quota:', selection['minimum_auto_count'])
print('Dự phòng:', sum(map(len, selection['reserves'].values())))


In [ ]:
# @title 3. Đọc MS1 và MS2_RESULTS.zip, tạo bundles (có thể chạy lại)
import json
from arxiv_pipeline.matching.silver_pool import prepare_pool
from arxiv_pipeline.matching.io import ms2_snapshot_path
MS1_PARENT = Path('/content/drive/MyDrive/arxiv_ms1_data') # @param {type:'string'}
MS2_PARENT = Path('/content/drive/MyDrive/arxiv_ms2_result') # @param {type:'string'}
ROOTS = Path('/content/drive/MyDrive/T3-05/roots.json')
def member_dir(parent, member):
    if not parent.is_dir():
        raise FileNotFoundError(f'Colab không thấy {parent}. Nếu là folder Được chia sẻ với tôi, thêm shortcut vào MyDrive.')
    matches = [p for p in parent.iterdir() if p.is_dir() and p.name.casefold() == member]
    if len(matches) != 1:
        raise FileNotFoundError(f'Cần đúng một folder {member} trong {parent}; tìm thấy {len(matches)}')
    return matches[0]
if not ROOTS.is_file():
    roots = {}
    for member in (f'tv{i}' for i in range(1, 6)):
        m1 = member_dir(MS1_PARENT, member)
        m1 = m1/'data' if (m1/'data').is_dir() else m1
        if not (m1/'state').is_dir() or not (m1/'references').is_dir():
            raise FileNotFoundError(f'MS1 thiếu state/ hoặc references/: {m1}')
        m2 = member_dir(MS2_PARENT, member)
        ms2_snapshot_path(m2) # xác nhận folder có state/ hoặc đúng một ZIP
        roots[member] = {'ms1_root': str(m1), 'ms2_root': str(m2)}
    ROOTS.parent.mkdir(parents=True, exist_ok=True)
    ROOTS.write_text(json.dumps(roots, indent=2), encoding='utf-8')
print('Nguồn:', ROOTS.read_text(encoding='utf-8'))
result = prepare_pool(Path(OUTPUT)/'selection.json', ROOTS, Path(OUTPUT)/'prepared')
print('Ready:', result['ready'], '| Chưa ready:', result['not_ready'])
print('Manifest:', Path(OUTPUT)/'prepared/prepared_manifest.json')
